# Exp3: CGM + time_since_meal + image (resnet18, gated)

In [2]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [11]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+resnet18 (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'resnet18'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_resnet18'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_resnet18_stability_summary.json'

In [8]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 512), (256, 5)]


In [9]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [10]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=512, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_resnet' /f'gated_cgm_time_resnet18_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_resnet18_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — CGM+time+resnet18 (gated) | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.17 | MAE 8.41 | gRMSE 13.76 | Zone A 96.71% | A+B 99.83%
 30 min | RMSE 18.81 | MAE 12.60 | gRMSE 22.18 | Zone A 91.04% | A+B 99.49%
 60 min | RMSE 27.82 | MAE 17.99 | gRMSE 34.51 | Zone A 82.26% | A+B 98.79%
 90 min | RMSE 33.52 | MAE 21.54 | gRMSE 42.44 | Zone A 76.13% | A+B 98.10%
120 min | RMSE 36.70 | MAE 23.90 | gRMSE 46.82 | Zone A 72.26% | A+B 97.71%

--------------------------------------------------------------------------------
Test results — CGM+time+resnet18 (gated) | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.80 | MAE 8.00 | gRMSE 13.01 | Zone A 96.46% | A+B 99.82%
 30 min | RMSE 18.51 | MAE 12.28 | gRMSE 21.41 | Zone A 90.59% | A+B 99.52%
 60 min | RMSE 27.42 | MAE 17.88 | gRMSE 33.55 | Zone A 81.69% | A+B 98.94%
 90 min | RMSE 33.17 | MAE 21.72 | gRMSE 41.61 | Zone A 75.15% | A+B 98.26%
120 min | RMSE 36.51 | MAE 24.11 | gRMSE 46.36 | Zone A 71.30% | A+B 97.80%

--------------------------------------------------------------------------------
Test results — CGM+time+resnet18 (gated) | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


...

 15 min | RMSE 11.73 | MAE 7.94 | gRMSE 12.87 | Zone A 96.30% | A+B 99.75%
 30 min | RMSE 18.29 | MAE 12.18 | gRMSE 21.06 | Zone A 90.44% | A+B 99.51%
 60 min | RMSE 27.14 | MAE 17.78 | gRMSE 33.06 | Zone A 81.45% | A+B 98.95%
 90 min | RMSE 32.84 | MAE 21.51 | gRMSE 40.99 | Zone A 75.48% | A+B 98.40%
120 min | RMSE 36.26 | MAE 23.99 | gRMSE 45.85 | Zone A 71.48% | A+B 98.01%

--------------------------------------------------------------------------------
Test results — CGM+time+resnet18 (gated) | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.57 | MAE 7.74 | gRMSE 13.20 | Zone A 97.10% | A+B 99.80%
 30 min | RMSE 18.63 | MAE 12.27 | gRMSE 21.99 | Zone A 90.46% | A+B 99.52%
 60 min | RMSE 27.81 | MAE 18.15 | gRMSE 34.26 | Zone A 80.87% | A+B 98.91%
 90 min | RMSE 33.44 | MAE 21.97 | gRMSE 42.08 | Zone A 74.87% | A+B 98.32%
120 min | RMSE 36.50 | MAE 24.17 | gRMSE 46.35 | Zone A 71.57% | A+B 97.95%

--------------------------------------------------------------------------------
Test results — CGM+time+resnet18 (gated) | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,260,757


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.71 | MAE 7.90 | gRMSE 13.30 | Zone A 96.99% | A+B 99.78%
 30 min | RMSE 18.50 | MAE 12.27 | gRMSE 21.90 | Zone A 90.96% | A+B 99.49%
 60 min | RMSE 27.34 | MAE 17.83 | gRMSE 33.86 | Zone A 81.66% | A+B 98.89%
 90 min | RMSE 32.84 | MAE 21.51 | gRMSE 41.36 | Zone A 75.45% | A+B 98.34%
120 min | RMSE 36.19 | MAE 23.90 | gRMSE 46.05 | Zone A 71.62% | A+B 97.86%

--------------------------------------------------------------------------------


In [12]:
# Test Results:
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [r['test_results'][h]['ceg']['A'] for r in all_results]
        elif m == 'zone_ab':
            seed_values = [r['test_results'][h]['ceg']['A'] + r['test_results'][h]['ceg']['B']
                            for r in all_results]
        else:
            seed_values = [r['test_results'][h][m] for r in all_results]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [_extract(r['test_results_per_participant'][pid][h], m)
                                    for r in all_results]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}

print(f'=== {MODEL_NAME} test set: overall mean +/- SD_seed (SD_participant) ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME, 'seeds': SEEDS, 'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS, 'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')

=== CGM+time+resnet18 (gated) test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.80 +/- 0.24 (1.99) mg/dL   MAE = 8.01 +/- 0.25 (1.54)   gRMSE = 13.23 +/- 0.34 (3.30)   Zone A = 96.69 +/- 0.32 (2.19)%   Zone A+B = 99.78 +/- 0.04 (0.34)%
 30-min:  RMSE = 18.61 +/- 0.24 (3.97) mg/dL   MAE = 12.36 +/- 0.18 (2.80)   gRMSE = 21.84 +/- 0.49 (6.75)   Zone A = 90.66 +/- 0.28 (3.40)%   Zone A+B = 99.50 +/- 0.02 (0.69)%
 60-min:  RMSE = 27.63 +/- 0.40 (9.23) mg/dL   MAE = 17.99 +/- 0.19 (5.74)   gRMSE = 34.08 +/- 0.75 (14.82)   Zone A = 81.57 +/- 0.47 (3.57)%   Zone A+B = 98.86 +/- 0.12 (1.13)%
 90-min:  RMSE = 33.31 +/- 0.43 (13.17) mg/dL   MAE = 21.73 +/- 0.19 (7.79)   gRMSE = 41.95 +/- 0.79 (20.78)   Zone A = 75.33 +/- 0.50 (5.82)%   Zone A+B = 98.22 +/- 0.20 (1.96)%
120-min:  RMSE = 36.61 +/- 0.37 (15.73) mg/dL   MAE = 24.09 +/- 0.18 (9.26)   gRMSE = 46.57 +/- 0.65 (24.67)   Zone A = 71.51 +/- 0.68 (7.38)%   Zone A+B = 97.80 +/- 0.20 (2.86)%

saved per-seed results t